# 05. Фиксация B0 и анализ ошибок NER

Ноутбук выполняет первые два этапа улучшения baseline:

1. регистрирует текущий запуск как неизменяемый baseline `B0`;
2. заново оценивает лучший checkpoint на **validation** и классифицирует ошибки.

Test здесь не используется для настройки решений.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import runpy

PROJECT_DIR = Path("/content/drive/MyDrive/NER_RuREBus_project")
EXPERIMENT_CONFIG = PROJECT_DIR / "configs" / "experiments" / "ner_baseline_v1.yaml"
OUTPUT_DIR = PROJECT_DIR / "results" / "ner_baseline" / "ner_baseline_v1"
BOOTSTRAP = PROJECT_DIR / "colab_bootstrap.py"

for required_path in (EXPERIMENT_CONFIG, OUTPUT_DIR / "checkpoints" / "best", BOOTSTRAP):
    if not required_path.exists():
        raise FileNotFoundError(f"Не найден {required_path}. Проверьте PROJECT_DIR и завершённость baseline.")

bootstrap_project = runpy.run_path(str(BOOTSTRAP))["bootstrap_project"]
bootstrap_project(PROJECT_DIR)

## 1. Регистрация baseline B0

Создаются `baseline_record.json` с SHA-256 данных, конфигурации, метрик и checkpoint и маркер `.baseline_locked`. Повторный запуск обучения в этот output-каталог будет запрещён.

In [ ]:
from rurebus_ie.training import register_ner_baseline

print("Хеширование checkpoint и артефактов B0; файл модели ~716 MB, это может занять некоторое время...")
baseline_record = register_ner_baseline(
    EXPERIMENT_CONFIG,
    alias="B0",
    project_root=PROJECT_DIR,
)
print("Baseline:", baseline_record["alias"])
print("Registered:", baseline_record["registered_at_utc"])
print("Artifacts with hashes:", len(baseline_record["artifacts"]))
print("Record:", OUTPUT_DIR / "baseline_record.json")

## 2. Validation inference и error analysis

Checkpoint не переобучается. Он только запускается на validation, после чего ошибки сопоставляются с золотыми сущностями.

In [ ]:
from rurebus_ie.training import run_validation_error_analysis

print("Запуск inference на validation без переобучения...")
analysis = run_validation_error_analysis(
    EXPERIMENT_CONFIG,
    project_root=PROJECT_DIR,
    edge_token_count=16,
)
summary = analysis.summary
print("Documents:", summary["documents"])
print("Gold entities:", summary["gold_entities"])
print("Predicted entities:", summary["predicted_entities"])
print(f"Validation strict micro-F1: {summary['strict_metrics']['micro_f1']:.4f}")

In [ ]:
import pandas as pd

strict_counts = pd.Series(summary["strict_counts"], name="count").to_frame()
diagnostic_counts = pd.Series(summary["diagnostic_categories"], name="count").to_frame()
per_class = pd.DataFrame(analysis.per_class).set_index("entity_type")
length_breakdown = pd.DataFrame(analysis.length_breakdown).set_index("token_bucket")

display(strict_counts)
display(diagnostic_counts)
display(per_class.sort_values("f1", ascending=False))
display(length_breakdown)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
diagnostic_counts.sort_values("count").plot.barh(ax=axes[0], legend=False, title="Категории ошибок")
per_class[["precision", "recall", "f1"]].sort_values("f1").plot.barh(ax=axes[1], title="Strict-метрики по классам")
axes[1].set_xlim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
confusion = pd.DataFrame(analysis.confusion)
if not confusion.empty:
    confusion_matrix = confusion.groupby(["gold_type", "predicted_type"])["count"].sum().unstack(fill_value=0)
    plt.figure(figsize=(10, 7))
    sns.heatmap(confusion_matrix, annot=True, fmt="d", cmap="Blues")
    plt.title("Ошибки типов: gold → prediction")
    plt.show()
    display(confusion.sort_values("count", ascending=False).head(30))
else:
    print("Ошибок типов не найдено.")

In [ ]:
errors = pd.DataFrame(analysis.errors)
boundary_errors = errors[errors["category"].isin(["boundary_error", "boundary_and_type_error"])]
false_positives = errors[errors["category"] == "false_positive"]
false_negatives = errors[errors["category"] == "false_negative"]

print("Boundary diagnostics:", summary["boundary_diagnostics"])
display(boundary_errors.sort_values(["iou", "confidence"], ascending=[False, False]).head(30))
display(false_positives.sort_values("confidence", ascending=False).head(30))
display(false_negatives.head(30))

In [ ]:
documents = pd.DataFrame(analysis.documents)
documents["strict_errors"] = documents["strict_false_positive"] + documents["strict_false_negative"]
display(documents.sort_values("strict_errors", ascending=False).head(25))

REPORT_DIR = OUTPUT_DIR / "error_analysis" / "validation"
print("Отчёт сохранён в:", REPORT_DIR)
for path in sorted(REPORT_DIR.iterdir()):
    print(" -", path.name)